### Create transaction before doing anything first :3

In [1]:
import pandas as pd
from collections import defaultdict
import os
import pickle
import time

def create_transactions_from_ratings(file_path, rating_threshold=3.5, save_path=None):
    """
    Convert movie ratings into transaction format for association rule mining
    
    Parameters:
    - file_path: Path to the rating.csv file
    - rating_threshold: Minimum rating to consider a movie as "liked" (default: 3.5)
    - save_path: Optional path to save the transactions for reuse
    
    Returns:
    - List of sets, where each set contains movie IDs that a user liked
    """
    start_time = time.time()
    print(f"Loading transactions from {file_path} with threshold {rating_threshold}...")
    
    # Check if saved transactions exist
    if save_path and os.path.exists(save_path):
        print(f"Loading pre-processed transactions from {save_path}...")
        with open(save_path, 'rb') as f:
            transaction_list = pickle.load(f)
        print(f"Loaded {len(transaction_list)} transactions in {time.time() - start_time:.2f} seconds")
        return transaction_list
    
    # read data in chunks
    chunks = pd.read_csv(file_path, chunksize=100000)
    
    transactions = defaultdict(set)
    total_ratings = 0
    
    for i, chunk in enumerate(chunks):
        if i >= 10:
            break
        print(f"Processing chunk {i+1}...")
        # only consider the ratings above threshold
        filtered_chunk = chunk[chunk['rating'] >= rating_threshold]
        
        for _, row in filtered_chunk.iterrows():
            try:
                transactions[row['userId']].add(row['movieId'].item())
            except:
               transactions[row['userId']].add(row['movieId'])
            
        total_ratings += len(chunk)
    
    transaction_list = list(transactions.values())
    
    transaction_list = [sorted(t) for t in transaction_list if len(t) > 0]
    
    print(f"Created {len(transaction_list)} transactions from {total_ratings} ratings")
    
    # Save transactions if path is provided
    if save_path:
        save_dir = os.path.dirname(save_path)
        if save_dir and not os.path.exists(save_dir):
            os.makedirs(save_dir)
        
        print(f"Saving transactions to {save_path}...")
        with open(save_path, 'wb') as f:
            pickle.dump(transaction_list, f)
    
    print(f"Total processing time: {time.time() - start_time:.2f} seconds")
    return transaction_list

In [2]:
transaction_list = create_transactions_from_ratings(r'/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/rating.csv', save_path='/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/transaction.pkl')

Loading transactions from /home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/rating.csv with threshold 3.5...
Loading pre-processed transactions from /home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/transaction.pkl...
Loaded 6738 transactions in 0.04 seconds


### Guide to run Apriori algorithm

In [5]:
from algorithms.apriori import Apriori
import pickle as pkl

# with open('/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_1m/transaction.pkl', 'rb') as f:
#     transaction_list = pkl.load(f)

apriori = Apriori(min_support=0.15, min_confidence=0.5)
apriori.fit(transaction_list)

Generating 1-itemsets
Generating 2-itemsets
Generating 3-itemsets
Generating 4-itemsets


In [ ]:
apriori.save_results('/home/hinhnv/Hai/KDLVKP/data_mining_code/data/movielens_20m/apriori_results.pkl')

### Guide to run Hashtree Apriori algorithm

In [ ]:
from algorithms.hashtree_apriori import HashTreeApriori

hash_tree_apriori = HashTreeApriori(min_support=0.15, min_confidence=0.5, max_leaf_size=64, max_depth=4)
hash_tree_apriori.fit(transaction_list)

Found 11952 unique items in 6738 transactions
Found 67 frequent 1-itemsets
Finding frequent 1-itemsets...
Finding frequent 2-itemsets...
Generated 2211 candidate 2-itemsets
Found 120 frequent 2-itemsets
Finding frequent 3-itemsets...
Generated 247 candidate 3-itemsets
Found 14 frequent 3-itemsets
Finding frequent 4-itemsets...
Generated 3 candidate 4-itemsets
Found 0 frequent 4-itemsets
Generated 223 association rules
Total runtime: 93.55 seconds


In [9]:
hash_tree_apriori.frequent_itemsets

201

### Guide to run Content-based filtering algorithm (quite computation-exhaustive)

#### Embedding movies & users using multiprocessing
We can use create_user_embedding method in utils.create_embedding to create user embeddings, but that implementation does not support multi-processing.

In [3]:
import numpy as np
import pandas as pd
import pickle
from tqdm import tqdm
import multiprocessing as mp # For multiprocessing

# Worker function for multiprocessing. Must be defined at the top level.
def process_user_embedding_task(user_id, user_specific_ratings_df, all_movie_embeddings, embedding_s):
    """Calculates embedding for a single user."""
    user_emb = np.zeros(embedding_s)
    denom = 0
    for _, rated_row in user_specific_ratings_df.iterrows():
        movie_id = rated_row['movieId']
        actual_rating = rated_row['rating']

        # IMPORTANT: Check if movie_id exists in movie_embeddings
        if movie_id in all_movie_embeddings:
            current_movie_embedding = all_movie_embeddings[movie_id]
            # Ensure it's a numpy array and has the expected shape
            if isinstance(current_movie_embedding, np.ndarray) and current_movie_embedding.shape == embedding_s:
                if np.any(current_movie_embedding): # Check if embedding is not all zeros
                    denom += 1
                    user_emb += actual_rating * current_movie_embedding
            # else: you might want to log if an embedding is malformed (e.g., wrong type or shape)
        # else: you might want to log if a movie_id from ratings is not in movie_embeddings

    if denom > 0:
        user_emb /= denom
    return user_id, user_emb

def create_user_embedding(ratings_file_path: str = 'data/movielens_20m/rating.csv', 
                          movie_embedding_path: str = 'data/movielens_20m/movie_embedding.pkl',
                          save_path: str = 'data/movielens_20m/user_embedding.pkl'):
    
    try:
        rating = pd.read_csv(ratings_file_path)
        # movies_df = pd.read_csv(movie_file_path) # Not directly used in current embedding logic
    except FileNotFoundError as e:
        print(f"Error: Could not read ratings file. {e}")
        return {}
    
    try:
        with open(movie_embedding_path, 'rb') as f:
            movie_embeddings = pickle.load(f)
    except FileNotFoundError:
        print(f"Error: Movie embedding file not found at {movie_embedding_path}")
        return {}
    except Exception as e:
        print(f"Error loading movie_embeddings: {e}")
        return {}
        
    user_embedding = {}
    
    # Robustly determine embedding shape
    _embedding_shape = None
    if not movie_embeddings or not isinstance(movie_embeddings, dict) or len(movie_embeddings) == 0:
        print("Error: movie_embeddings is empty, not a dictionary, or invalid. Cannot proceed.")
        return {}
    
    # Try to get shape from the first valid NumPy array embedding
    for key in movie_embeddings: # Iterate to find the first valid embedding
        if isinstance(movie_embeddings[key], np.ndarray):
            _embedding_shape = movie_embeddings[key].shape
            break 
    
    if _embedding_shape is None:
        print("Error: Could not determine a valid embedding shape from movie_embeddings (no NumPy arrays found as values).")
        return {}
    
    print(f"Determined embedding shape: {_embedding_shape}")

    # Prepare arguments for each task for multiprocessing
    tasks_args = []
    # Grouping by userId is efficient as each worker gets only the data it needs for one user
    grouped_ratings = rating.groupby('userId')
    for user_id_val, group_df in grouped_ratings:
        tasks_args.append((user_id_val, group_df, movie_embeddings, _embedding_shape))

    if not tasks_args:
        print("No users found in ratings data to process.")
        return {}

    # Determine number of processes
    # Using mp.cpu_count() can be aggressive; mp.cpu_count() // 2 or mp.cpu_count() - 1 is often a good start
    num_processes = max(1, mp.cpu_count() // 2) 
    print(f"Starting user embedding computation with {num_processes} processes for {len(tasks_args)} users...")

    results = []
    # Create a pool of worker processes
    # The `if __name__ == "__main__":` guard is crucial for multiprocessing on some OS (like Windows)
    # when running scripts. For Jupyter notebooks, it's usually not needed for the pool itself,
    # but the worker function must be defined at the top-level.
    try:
        with mp.Pool(processes=num_processes) as pool:
            # Use tqdm for progress bar. pool.starmap executes tasks and returns results in order.
            results = list(tqdm(pool.starmap(process_user_embedding_task, tasks_args), total=len(tasks_args), desc="Processing users"))
    except Exception as e:
        print(f"An error occurred during multiprocessing: {e}")
        print("Consider if the worker function or data being passed is picklable.")
        # Optionally, you could add a fallback to single-threaded processing here for debugging:
        # print("Falling back to single-threaded processing due to error.")
        # results = []
        # for args_tuple in tqdm(tasks_args, desc="Single-threaded fallback"):
        #     results.append(process_user_embedding_task(*args_tuple))


    for res_user_id, res_user_emb in results:
        user_embedding[res_user_id] = res_user_emb

    print(f"Finished processing embeddings for {len(user_embedding)} users.")
    
    if save_path:
        try:
            with open(save_path, 'wb') as f:
                pickle.dump(user_embedding, f)
                print(f"User embeddings saved to {save_path}")
        except Exception as e:
            print(f"Error saving user embeddings to {save_path}: {e}")
    
    return user_embedding


In [4]:
from utils.create_embedding import create_movie_embedding
movie_embeddings_path = 'data/movielens_20m/movie_embedding.pkl'
user_embeddings_path = 'data/movielens_20m/user_embedding.pkl'
movie_embeddings = create_movie_embedding(movies_file_path='data/movielens_20m/movies_processed.csv', 
                                          model_name='all-MiniLM-L6-v2', 
                                          save_path=movie_embeddings_path)

user_embeddings = create_user_embedding(ratings_file_path='data/movielens_20m/rating.csv', 
                                        movie_embedding_path='data/movielens_20m/movie_embedding.pkl',
                                        save_path=user_embeddings_path)


/home/hinhnv/Hai/KDLVKP/data_mining_code/kdlvkp_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating movie embeddings for 1127 tags...


100%|██████████| 1127/1127 [00:08<00:00, 129.63it/s]


Creating movie embeddings for 27278 movies...


27278it [00:00, 45292.54it/s]


Movie embeddings saved to data/movielens_20m/movie_embedding.pkl
Determined embedding shape: (384,)
Starting user embedding computation with 28 processes for 138493 users...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Finished processing embeddings for 138493 users.
User embeddings saved to data/movielens_20m/user_embedding.pkl


In [31]:
movie_embds

array([[-0.03645131,  0.01606606, -0.0008699 , ...,  0.01498496,
         0.02105172,  0.01798541],
       [-0.03915351,  0.02080706, -0.01142101, ...,  0.00309874,
         0.00574298,  0.02417969],
       [-0.04526789,  0.00648963, -0.01797347, ...,  0.00176279,
         0.03798633,  0.01792262],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], shape=(27278, 384))

In [32]:
import os
import sys
from algorithms.content_based_filtering import ContentBasedFiltering
content_based_filtering = ContentBasedFiltering(item_embedding=np.vstack([val for _, val in movie_embeddings.items()]),
                                                user_embedding=np.vstack([val for _, val in user_embeddings.items()]))

In [33]:
movie_embedding_keys = list(movie_embeddings.keys())
user_embedding_keys = list(user_embeddings.keys())

In [16]:
ratings_df = pd.read_csv('data/movielens_20m/rating.csv')
movies_df = pd.read_csv('data/movielens_20m/movie.csv')

In [1]:
import pandas as pd

def get_top_rated_movies_for_user(user_id, top_n=10):
    """
    Retrieve the top N movies rated by a specific user along with movie information.
    
    Parameters:
    -----------
    user_id : int
        The ID of the user
    top_n : int, optional
        Number of top rated movies to return (default: 10)
        
    Returns:
    --------
    DataFrame containing the user's top rated movies with movie information
    """
    global ratings_df, movies_df
    

    user_ratings = ratings_df[ratings_df['userId'] == user_id]
    user_ratings = user_ratings.sort_values(by='rating', ascending=False)
    
    top_rated = user_ratings.head(top_n)

    result = pd.merge(top_rated, movies_df, on='movieId')
    result = result[['movieId', 'title', 'genres', 'rating']]
    
    return result

In [48]:
top_movies = get_top_rated_movies_for_user(user_id=1, top_n=10)
top_movies

,movieId,title,genres,rating
0,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,5.0
1,8507,Freaks (1932),Crime|Drama|Horror,5.0
2,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,5.0
3,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,5.0
4,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,4.5
5,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,4.5
6,8636,Spider-Man 2 (2004),Action|Adventure|Sci-Fi|IMAX,4.5
7,1036,Die Hard (1988),Action|Crime|Thriller,4.0
8,1200,Aliens (1986),Action|Adventure|Horror|Sci-Fi,4.0
9,1097,E.T. the Extra-Terrestrial (1982),Children|Drama|Sci-Fi,4.0


In [49]:
recommended_movies_ids = content_based_filtering.recommend(user_id=1, top_k=50)

/home/hinhnv/Hai/KDLVKP/data_mining_code/utils/metrics.py:14: RuntimeWarning: invalid value encountered in divide
  return a @ b.T / (np.linalg.norm(a, axis=1, keepdims=True) * np.linalg.norm(b, axis=1, keepdims=True).T)


In [50]:
movie_ids = [movie_embedding_keys[i] for i in recommended_movies_ids]

In [51]:
movies_df[movies_df['movieId'].isin(movie_ids)]

,movieId,title,genres
14872,74475,"Headless Woman, The (Mujer sin cabeza, La) (2008)",Drama|Mystery|Thriller
14873,74478,Yellow (1998),Comedy|Drama
14874,74480,"Other End of the Line, The (2008)",Comedy|Romance
14878,74491,1066 (2009),Action|Adventure|War
14879,74493,"Steam Experiment, The (2009)",Drama|Thriller
14881,74506,"Bakery Girl of Monceau, The (La boulangère de ...",Romance
14884,74512,Steal This Film (2006),Documentary
14887,74538,Mademoiselle (1966),Drama
14890,74547,Darling Lili (1970),Drama|War
14892,74573,"Vicious Kind, The (2009)",Comedy|Drama


### Colaborative filtering

In [1]:
import numpy as np
import pandas as pd
from utils.preprocess import create_user_item_matrix
from algorithms.colab_filtering import ColabFiltering

# Attempt to load existing data
user_item_matrix = create_user_item_matrix(
    ratings_file='data/movielens_20m/ratings.csv',
    movies_file='data/movielens_20m/movies.csv'
)

/home/hinhnv/Hai/KDLVKP/data_mining_code/kdlvkp_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
user_item_matrix

array([[nan, 3.5, nan, ..., nan, nan, nan],
       [nan, nan, 4. , ..., nan, nan, nan],
       [4. , nan, nan, ..., nan, nan, nan],
       ...,
       [4. , nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [3.5, nan, nan, ..., nan, nan, nan]], shape=(702, 8227))

In [3]:
# Strategies: 'user_based', 'item_based'
# Similarity metrics: 'cosine', 'pearson'
k_neighbors = 5
colab_filter_user_based_cosine = ColabFiltering(strategy='item_based', similarity_metric='cosine', k=k_neighbors)


In [4]:
colab_filter_user_based_cosine.fit(user_item_matrix)

/home/hinhnv/Hai/KDLVKP/data_mining_code/algorithms/colab_filtering.py:82: RuntimeWarning: invalid value encountered in divide
  pred = np.where(denom>0, nom/denom, -1)
/home/hinhnv/Hai/KDLVKP/data_mining_code/algorithms/colab_filtering.py:82: RuntimeWarning: divide by zero encountered in divide
  pred = np.where(denom>0, nom/denom, -1)


In [5]:
colab_filter_user_based_cosine.user_item_matrix

array([[-1.        ,  3.5       , -1.        , ..., -1.        ,
         0.01380936, -1.        ],
       [ 5.        ,  5.        ,  4.        , ..., -1.        ,
        -1.        , -1.        ],
       [ 4.        ,  5.        ,  4.        , ..., -1.        ,
        -1.        , -1.        ],
       ...,
       [ 4.        ,  3.86430681, -1.        , ..., -1.        ,
        -1.        , -1.        ],
       [-1.        ,  1.        , -1.        , ..., -1.        ,
        -1.        , -1.        ],
       [ 3.5       ,  2.51114666, -1.        , ..., -1.        ,
        -1.        , -1.        ]], shape=(702, 8227))